In [1]:
pip show langchain

Name: langchain
Version: 1.2.9
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: d:\llm_langchain\chatbot_langchain\langchain_env\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Import a loader that reads .docx files and converts them into LangChain Documents
from langchain_community.document_loaders import Docx2txtLoader

# Import a splitter that can split text based on Markdown headers (#, ##, etc.)
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter

# Import a splitter that splits text into fixed-size character chunks
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

In [9]:
# --------------------------------------------------
# 1️⃣ Load the DOCX file
# --------------------------------------------------

# We use Docx2txtLoader to:
# - Extract text from a Microsoft Word (.docx) file
# - Convert it into LangChain Document objects
# - Preserve metadata for later use (important for RAG systems)

loader_docx = Docx2txtLoader("Introduction_to_Data_and_Data_Science_2.docx")

# load() returns a list of Document objects
# Each Document contains:
# - page_content (the actual text)
# - metadata (source info)
pages = loader_docx.load()


# --------------------------------------------------
# 2️⃣ Split by Markdown headers (Semantic Splitting)
# --------------------------------------------------

# MarkdownHeaderTextSplitter is used when:
# - Your document has structure (like # Title, ## Subtitle)
# - You want chunks grouped by logical sections
# - This keeps related content together (better embeddings!)

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Course Title"),      # Split when single # is found
        ("##", "Lecture Title")     # Split when double ## is found
    ]
)

# We split the first page content into structured sections
pages_md_split = md_splitter.split_text(pages[0].page_content)


# --------------------------------------------------
# 3️⃣ Clean the text (Remove extra spaces & newlines)
# --------------------------------------------------

# Why?
# - Extra spaces and newlines create noisy embeddings
# - Cleaning improves vector quality
# - Makes chunking more consistent

for i in range(len(pages_md_split)):
    pages_md_split[i].page_content = ' '.join(
        pages_md_split[i].page_content.split()
    )


# --------------------------------------------------
# 4️⃣ Character-based Chunk Splitting
# --------------------------------------------------

# Even after header splitting, sections may still be too large.
# LLMs have token limits, so we:
# - Break text into smaller chunks
# - Add overlap so context isn't lost between chunks

char_splitter = CharacterTextSplitter(
    separator=".",      # Split around sentences (better than random split)
    chunk_size=500,     # Max characters per chunk
    chunk_overlap=50    # 50 characters overlap to preserve context
)

# Split structured sections into final chunks
pages_char_split = char_splitter.split_documents(pages_md_split)

In [7]:
pages_char_split #Every Document is chunk file

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individu

In [10]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
pages_char_split[18]

Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='More importantly, it will be sufficient for your need to create quick and accurate analyses. However, if your theoretical preparation is strong enough, you will find yourself restricted by software. Knowing a programming language such as R and Python, gives you the freedom to create specific, ad-hoc tools for each project you are working on')

In [12]:
# Convert chunk 3 into a numerical vector representation
# embed_query() transforms text into embeddings (dense numerical vectors)
# These vectors capture the semantic meaning of the text
vector1 = embedding.embed_query(pages_char_split[3].page_content)

# Convert chunk 5 into a vector
# Now we can mathematically compare chunk 3 and chunk 5
# If their vectors are similar, the content is semantically similar
vector2 = embedding.embed_query(pages_char_split[5].page_content)

# Convert chunk 18 into a vector
# Each chunk becomes a point in high-dimensional vector space
# This allows similarity search using cosine similarity or dot product
vector3 = embedding.embed_query(pages_char_split[18].page_content)

In [13]:
len(vector1), len(vector2), len(vector3)

(384, 384, 384)

In [14]:

# Compute similarity between vector1 and vector2
# np.dot() calculates the dot product between two vectors
# The dot product gives a numerical score of similarity
# Higher value → more similar meaning
#------------------------------------------------------------
# Compute similarity between vector1 and vector3
# If the score is lower than similarity_1_2,
# it means chunk 1 and chunk 3 are less semantically related
#------------------------------------------------------------
# Compute similarity between vector2 and vector3
# This helps compare all chunks pairwise

np.dot(vector1, vector2), np.dot(vector1, vector3), np.dot(vector2, vector3)

(np.float64(0.371555368407836),
 np.float64(0.36037421626382166),
 np.float64(0.134796629102236))

In [15]:
# Calculate the magnitude (length) of vector1
# np.linalg.norm() computes the Euclidean length of the vector
# This tells us how "large" the vector is in vector space
#----------------------------------------------------------------
# Calculate the magnitude of vector2
# The norm is needed when computing cosine similarity
#----------------------------------------------------------------
# Calculate the magnitude of vector3
# It represents the distance of the vector from the origin (0,0,...,0)
np.linalg.norm(vector1), np.linalg.norm(vector2), np.linalg.norm(vector3)


(np.float64(1.0000000315600153),
 np.float64(1.0000000462160448),
 np.float64(1.0000000055022173))